# OneHotEncoder Examples in Scikit-Learn

This notebook gives simple examples of `OneHotEncoder`.

`OneHotEncoder` is used to convert categorical text values such as city names, colors, car types, or wine color into numeric columns of `0` and `1`.

Machine learning models need numbers, so categorical columns must be encoded before fitting the model.

## 1. Import the required libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression

## 2. Example 1: OneHotEncoder with one column

In this example, the `color` column contains text values.  
We will convert each color into a separate column.

In [3]:
df_color = pd.DataFrame({
    "color": ["red", "blue", "green", "red", "blue"]
})

df_color

,color
0,red
1,blue
2,green
3,red
4,blue


In [4]:
encoder = OneHotEncoder(sparse_output=False)

encoded_color = encoder.fit_transform(df_color[["color"]])

encoded_color_df = pd.DataFrame(
    encoded_color,
    columns=encoder.get_feature_names_out(["color"])
)

encoded_color_df

,color_blue,color_green,color_red
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0


Explanation:

Each color became a separate column.

For example:

- `red` becomes `color_red = 1`
- `blue` becomes `color_blue = 1`
- `green` becomes `color_green = 1`

All other columns are set to `0`.

## 3. Example 2: Encoding more than one categorical column

Now we encode two columns: `city` and `car_type`.

In [5]:
df_cars = pd.DataFrame({
    "city": ["Riyadh", "Jeddah", "Dammam", "Riyadh", "Jeddah"],
    "car_type": ["SUV", "Sedan", "SUV", "Truck", "Sedan"]
})

df_cars

,city,car_type
0,Riyadh,SUV
1,Jeddah,Sedan
2,Dammam,SUV
3,Riyadh,Truck
4,Jeddah,Sedan


In [6]:
encoder = OneHotEncoder(sparse_output=False)

encoded_cars = encoder.fit_transform(df_cars)

encoded_cars_df = pd.DataFrame(
    encoded_cars,
    columns=encoder.get_feature_names_out()
)

encoded_cars_df

,city_Dammam,city_Jeddah,city_Riyadh,car_type_SUV,car_type_Sedan,car_type_Truck
0,0.0,0.0,1.0,1.0,0.0,0.0
1,0.0,1.0,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,1.0,0.0


Explanation:

The encoder created columns for each unique category in both columns.

For example:

- `city_Riyadh`
- `city_Jeddah`
- `city_Dammam`
- `car_type_SUV`
- `car_type_Sedan`
- `car_type_Truck`

## 4. Example 3: Train and test data with unknown category

This example is important.

Sometimes the test data contains a category that did not appear in the training data.  
For example, the model trained on `Riyadh`, `Jeddah`, and `Dammam`, but later it sees `Makkah`.

To avoid an error, we use:

```python
handle_unknown="ignore"
```

In [7]:
train = pd.DataFrame({
    "city": ["Riyadh", "Jeddah", "Dammam", "Riyadh"]
})

test = pd.DataFrame({
    "city": ["Jeddah", "Riyadh", "Makkah"]
})

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

train_encoded = encoder.fit_transform(train)
test_encoded = encoder.transform(test)

train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoder.get_feature_names_out(["city"])
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoder.get_feature_names_out(["city"])
)

print("Train encoded:")
display(train_encoded_df)

print("Test encoded:")
display(test_encoded_df)

Train encoded:


,city_Dammam,city_Jeddah,city_Riyadh
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0


Test encoded:


,city_Dammam,city_Jeddah,city_Riyadh
0,0.0,1.0,0.0
1,0.0,0.0,1.0
2,0.0,0.0,0.0


Explanation:

`Makkah` was not in the training data.  
Because we used `handle_unknown="ignore"`, the encoder did not crash.  
It represented `Makkah` as all zeros because it does not know this category.

## 5. Example 4: Dataset with categorical and numeric columns

Most real datasets contain both categorical and numeric columns.

Here:

- `city` and `car_type` are categorical
- `age` and `salary` are numeric

We will encode only the categorical columns and keep the numeric columns.

In [8]:
df_customers = pd.DataFrame({
    "city": ["Riyadh", "Jeddah", "Dammam", "Riyadh", "Jeddah"],
    "car_type": ["SUV", "Sedan", "SUV", "Truck", "Sedan"],
    "age": [25, 30, 35, 40, 45],
    "salary": [8000, 10000, 12000, 15000, 17000]
})

df_customers

,city,car_type,age,salary
0,Riyadh,SUV,25,8000
1,Jeddah,Sedan,30,10000
2,Dammam,SUV,35,12000
3,Riyadh,Truck,40,15000
4,Jeddah,Sedan,45,17000


In [20]:
categorical_cols = ["city", "car_type"]

#Because dataset has mixed columns:
#ColumnTransformer lets apply different processing to different columns.
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(sparse_output=False), categorical_cols)
    ],
    remainder="passthrough" )

#"cat"                        = name of this step
#OneHotEncoder(...)            = encode text categories into 0/1 columns
#categorical_cols              = apply it only on city and car_type

encoded_customers = preprocessor.fit_transform(df_customers)

feature_names = list(
    preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
) + ["age", "salary"]

encoded_customers_df = pd.DataFrame(encoded_customers, columns=feature_names)

encoded_customers_df

,city_Dammam,city_Jeddah,city_Riyadh,car_type_SUV,car_type_Sedan,car_type_Truck,age,salary
0,0.0,0.0,1.0,1.0,0.0,0.0,25.0,8000.0
1,0.0,1.0,0.0,0.0,1.0,0.0,30.0,10000.0
2,1.0,0.0,0.0,1.0,0.0,0.0,35.0,12000.0
3,0.0,0.0,1.0,0.0,0.0,1.0,40.0,15000.0
4,0.0,1.0,0.0,0.0,1.0,0.0,45.0,17000.0


Explanation:

The categorical columns were converted into 0/1 columns.  
The numeric columns `age` and `salary` stayed as normal numbers.

## 6. Example 5: Using OneHotEncoder inside a machine learning model

This is the correct way to use `OneHotEncoder` in a model.

We will create a simple model that predicts whether a customer will buy or not.

In [10]:
df_buy = pd.DataFrame({
    "city": ["Riyadh", "Jeddah", "Dammam", "Riyadh", "Jeddah", "Dammam", "Riyadh", "Jeddah"],
    "car_type": ["SUV", "Sedan", "SUV", "Truck", "Sedan", "Truck", "SUV", "Truck"],
    "age": [25, 31, 45, 22, 36, 40, 29, 52],
    "buy": [1, 0, 1, 0, 1, 0, 1, 0]
})

df_buy

,city,car_type,age,buy
0,Riyadh,SUV,25,1
1,Jeddah,Sedan,31,0
2,Dammam,SUV,45,1
3,Riyadh,Truck,22,0
4,Jeddah,Sedan,36,1
5,Dammam,Truck,40,0
6,Riyadh,SUV,29,1
7,Jeddah,Truck,52,0


In [21]:
X = df_buy[["city", "car_type", "age"]]
y = df_buy["buy"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["city", "car_type"])
    ],
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    LogisticRegression()
)

model.fit(X, y)

new_customer = pd.DataFrame({
    "city": ["Riyadh"],
    "car_type": ["SUV"],
    "age": [29]
})

prediction = model.predict(new_customer)

print("Prediction:", prediction[0])

Prediction: 1


Explanation:

The pipeline first encodes `city` and `car_type`.  
Then it sends the transformed data to `LogisticRegression`.

The result is:

- `1` means the customer may buy
- `0` means the customer may not buy

## 7. Example 6: Wine Quality dataset example

In the Wine Quality dataset, a column such as `color` can contain:

- red
- white

We can encode it using `OneHotEncoder`.

In [12]:
df_wine_example = pd.DataFrame({
    "color": ["red", "white", "red", "white", "white"],
    "alcohol": [10.2, 11.5, 9.8, 12.1, 10.9],
    "quality": [5, 6, 5, 7, 6]
})

df_wine_example

,color,alcohol,quality
0,red,10.2,5
1,white,11.5,6
2,red,9.8,5
3,white,12.1,7
4,white,10.9,6


In [13]:
encoder = OneHotEncoder(sparse_output=False)

encoded_color = encoder.fit_transform(df_wine_example[["color"]])

color_df = pd.DataFrame(
    encoded_color,
    columns=encoder.get_feature_names_out(["color"])
)

final_wine_df = pd.concat(
    [df_wine_example.drop(columns=["color"]), color_df],
    axis=1
)

final_wine_df

,alcohol,quality,color_red,color_white
0,10.2,5,1.0,0.0
1,11.5,6,0.0,1.0
2,9.8,5,1.0,0.0
3,12.1,7,0.0,1.0
4,10.9,6,0.0,1.0


Explanation:

The text column `color` was replaced with:

- `color_red`
- `color_white`

This makes the data ready for machine learning.

## 8. Example 7: Regression model with encoded category

Here we use the wine color and alcohol value to predict wine quality.

This is only a small example for learning.

In [14]:
X = df_wine_example[["color", "alcohol"]]
y = df_wine_example["quality"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["color"])
    ],
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    LinearRegression()
)

model.fit(X, y)

new_wine = pd.DataFrame({
    "color": ["white"],
    "alcohol": [11.8]
})

predicted_quality = model.predict(new_wine)

print("Predicted quality:", predicted_quality[0])

Predicted quality: 6.558333333333333


Explanation:

The model cannot understand `white` directly.  
The pipeline encodes `white` first, then the model uses the encoded color and alcohol value to predict quality.

## 9. Example 8: Sparse output

By default, `OneHotEncoder` can return a sparse matrix.

A sparse matrix stores only the non-zero values.  
This saves memory when there are many categories.

In [15]:
df_large_categories = pd.DataFrame({
    "city": ["Riyadh", "Jeddah", "Dammam", "Makkah", "Madinah", "Riyadh"]
})

encoder = OneHotEncoder(sparse_output=True)

sparse_result = encoder.fit_transform(df_large_categories[["city"]])

print(type(sparse_result))
print(sparse_result)

<class 'scipy.sparse._csr.csr_matrix'>
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (6, 5)>
  Coords	Values
  (0, 4)	1.0
  (1, 1)	1.0
  (2, 0)	1.0
  (3, 3)	1.0
  (4, 2)	1.0
  (5, 4)	1.0


Explanation:

The output is not a normal DataFrame.  
It is a sparse matrix. It stores only the positions where the value is not zero.

This is useful when there are many categories because most values are zeros.

## 10. Example 9: Convert sparse output to normal table

Sometimes we want to see the sparse result as a normal table.

In [16]:
dense_result = sparse_result.toarray()

dense_df = pd.DataFrame(
    dense_result,
    columns=encoder.get_feature_names_out(["city"])
)

dense_df

,city_Dammam,city_Jeddah,city_Madinah,city_Makkah,city_Riyadh
0,0.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0,0.0
5,0.0,0.0,0.0,0.0,1.0


Explanation:

`.toarray()` converts the sparse matrix into a normal array.

This is useful for viewing the result, but for very large datasets, keeping sparse format is better because it saves memory.

## 11. Example 10: Dropping one category

Sometimes we use:

```python
drop="first"
```

This removes one category to avoid duplicated information.

For example, if we have `red` and `white`, we do not always need both columns.  
If `color_white = 0`, then the wine is red.

In [17]:
df_binary = pd.DataFrame({
    "color": ["red", "white", "red", "white"]
})

encoder = OneHotEncoder(sparse_output=False, drop="first")

encoded_binary = encoder.fit_transform(df_binary[["color"]])

encoded_binary_df = pd.DataFrame(
    encoded_binary,
    columns=encoder.get_feature_names_out(["color"])
)

encoded_binary_df

,color_white
0,0.0
1,1.0
2,0.0
3,1.0


Explanation:

Only one column is created.  
This is common in regression models because it avoids unnecessary duplicate columns.

## 12. Example 11: OneHotEncoder with pandas output

Scikit-Learn can return the encoded output directly as a pandas DataFrame by using:

```python
set_output(transform="pandas")
```

In [18]:
df_animals = pd.DataFrame({
    "animal": ["cat", "dog", "bird", "cat"],
    "size": ["small", "medium", "small", "large"]
})

encoder = OneHotEncoder(sparse_output=False)
encoder.set_output(transform="pandas")

encoded_animals = encoder.fit_transform(df_animals)

encoded_animals

,animal_bird,animal_cat,animal_dog,size_large,size_medium,size_small
0,0.0,1.0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,1.0,0.0,0.0


Explanation:

This is a clean way to get a DataFrame directly without manually creating it.

## 13. Final summary

`OneHotEncoder` is useful when categorical data needs to be used in machine learning.

Main points:

- It converts text categories into numeric columns.
- Each category becomes a separate 0/1 column.
- `handle_unknown="ignore"` helps avoid errors with new categories.
- `ColumnTransformer` is useful when the dataset has categorical and numeric columns together.
- Sparse output saves memory when there are many categories.